This is my exploratory data analysis of the AACT (Aggregate Analysis of ClinicalTrials.gov) using PostgreSQL. My goal with this project is to investigate the prevalence of trials relating to antimicrobial resistance (AMR) over the past few decades. Specifically before and after the World Health Organization adopted their global action plan surrounding AMR in 2015.

In [3]:
%load_ext sql


Load postgresSQL engine and begin to confirm tables exist

In [2]:


import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)


with engine.connect() as conn:
    query = text("""SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'ctgov'
LIMIT 20;""")
    
    
    test = pd.read_sql(query, conn)
    print(test)





                  table_name
0          browse_conditions
1      all_browse_conditions
2       browse_interventions
3   all_browse_interventions
4                 facilities
5                 all_cities
6                 conditions
7             all_conditions
8                  countries
9              all_countries
10           design_outcomes
11       all_design_outcomes
12            all_facilities
13             design_groups
14           all_group_types
15            id_information
16        all_id_information
17             interventions
18    all_intervention_types
19         all_interventions


Check on table row counts

In [2]:
with engine.connect() as conn:
    query = text("""
    SELECT 
    'studies' as table_name, COUNT(*) as row_count FROM ctgov.studies
UNION ALL
SELECT 'conditions', COUNT(*) FROM ctgov.conditions
UNION ALL
SELECT 'interventions', COUNT(*) FROM ctgov.interventions
UNION ALL
SELECT 'sponsors', COUNT(*) FROM ctgov.sponsors
UNION ALL
SELECT 'facilities', COUNT(*) FROM ctgov.facilities;
    """)
    
    test = pd.read_sql(query, conn)

print(test)

      table_name  row_count
0       sponsors     932641
1  interventions     987470
2     conditions    2085810
3     facilities    3446062
4        studies    1168202


Begin first AMR cohort filtering, just filtering for keywords typically relating to AMR

In [7]:
with engine.connect() as conn:
    query = text("""
    SELECT 
    s.nct_id,
    s.brief_title,
    s.overall_status,
    s.phase,
    s.start_date,
    s.completion_date,
    s.enrollment,
    sp.name AS sponsor_name,
    sp.lead_or_collaborator
FROM ctgov.studies s
JOIN ctgov.sponsors sp ON s.nct_id = sp.nct_id
JOIN ctgov.conditions c ON s.nct_id = c.nct_id
WHERE 
    c.downcase_name ILIKE '%drug resistant%'
    OR c.downcase_name ILIKE '%antimicrobial resistance%'
    OR c.downcase_name ILIKE '%multidrug resistant%'
    OR c.downcase_name ILIKE '%MRSA%'
    OR c.downcase_name ILIKE '%tuberculosis%'
LIMIT 100;
    """)
    
    test = pd.read_sql(query, conn)

print(test)
print(f"Unique trials: {test['nct_id'].nunique()}")



         nct_id                                        brief_title  \
0   NCT06917495  Short-Course Anti-tuberculosis Regimens for Mi...   
1   NCT06917495  Short-Course Anti-tuberculosis Regimens for Mi...   
2   NCT06917495  Short-Course Anti-tuberculosis Regimens for Mi...   
3   NCT06917495  Short-Course Anti-tuberculosis Regimens for Mi...   
4   NCT00595907  Results of an Interferon-Gamma Release Assay A...   
..          ...                                                ...   
95  NCT07178860  Respiratory Health and Wellbeing Post Pulmonar...   
96  NCT07178860  Respiratory Health and Wellbeing Post Pulmonar...   
97  NCT03220464  Early Diagnosis of Active Tuberculosis Using U...   
98  NCT03220464  Early Diagnosis of Active Tuberculosis Using U...   
99  NCT03220464  Early Diagnosis of Active Tuberculosis Using U...   

        overall_status   phase  start_date completion_date  enrollment  \
0   NOT_YET_RECRUITING  PHASE2  2025-04-05      2029-12-31         300   
1   NOT_YET

Check what "resistant" conditions actually exist

In [8]:
with engine.connect() as conn:
    query = text("""
SELECT DISTINCT downcase_name, COUNT(*) as trial_count
FROM ctgov.conditions
WHERE downcase_name ILIKE '%resistant%'
GROUP BY downcase_name
ORDER BY trial_count DESC
LIMIT 50;""")
    
    test = pd.read_sql(query, conn)

print(test)

                                        downcase_name  trial_count
0                      treatment resistant depression          428
1     metastatic castration-resistant prostate cancer          380
2                castration-resistant prostate cancer          200
3             castration-resistant prostate carcinoma          148
4                             drug resistant epilepsy          142
5                              resistant hypertension          136
6                      antibiotic resistant infection          124
7                   hormone-resistant prostate cancer          110
8            depressive disorder, treatment-resistant          108
9     metastatic castration resistant prostate cancer           90
10        methicillin-resistant staphylococcus aureus           88
11                  platinum-resistant ovarian cancer           82
12          prostatic neoplasms, castration-resistant           82
13                     treatment-resistant depression         

There are many trials about "treatment-resistance" which is not what we are after. We need to hone in on antimicrobial resistance. Going to add a dual AND filter to capture specific pathogens and groups of pathogens relating to AMR

In [20]:
with engine.connect() as conn:
    query = text("""
SELECT DISTINCT downcase_name, COUNT(*) as trial_count
FROM ctgov.conditions
WHERE 
    (
        downcase_name ILIKE '%-resistant%'
        OR downcase_name ILIKE '%drug resistant%'
        OR downcase_name ILIKE '%antimicrobial resistance%'
        OR downcase_name ILIKE '%antibiotic resistance%'
    )
    AND
    (
        downcase_name ILIKE '%staphylococcus%'
        OR downcase_name ILIKE '%tuberculosis%'
        OR downcase_name ILIKE '%enterobacteriaceae%'
        OR downcase_name ILIKE '%pseudomonas%'
        OR downcase_name ILIKE '%klebsiella%'
        OR downcase_name ILIKE '%acinetobacter%'
        OR downcase_name ILIKE '%enterococcus%'
        OR downcase_name ILIKE '%difficile%'
        OR downcase_name ILIKE '%escherichia%'
        OR downcase_name ILIKE '%campylobacter%'
        OR downcase_name ILIKE '%candida%'
        OR downcase_name ILIKE '%mrsa%'
        OR downcase_name ILIKE '%malaria%'
        OR downcase_name ILIKE '%infection%'
        OR downcase_name ILIKE '%bacterial%'
        OR downcase_name ILIKE '%fungal%'
        OR downcase_name ILIKE '%antimicrobial%'
        OR downcase_name ILIKE '%antifungal%'
        OR downcase_name ILIKE '%antibiotic%'
    )
GROUP BY downcase_name
ORDER BY trial_count DESC
LIMIT 50;""")
    
    test = pd.read_sql(query, conn) 
print(test) 

with engine.connect() as conn:
    query = text("""
SELECT COUNT(DISTINCT downcase_name) AS unique_condition_count
FROM ctgov.conditions
WHERE 
    (
        downcase_name ILIKE '%-resistant%'
        OR downcase_name ILIKE '%drug resistant%'
        OR downcase_name ILIKE '%antimicrobial resistance%'
        OR downcase_name ILIKE '%antibiotic resistance%'
    )
    AND
    (
        downcase_name ILIKE '%staphylococcus%'
        OR downcase_name ILIKE '%tuberculosis%'
        OR downcase_name ILIKE '%enterobacteriaceae%'
        OR downcase_name ILIKE '%pseudomonas%'
        OR downcase_name ILIKE '%klebsiella%'
        OR downcase_name ILIKE '%acinetobacter%'
        OR downcase_name ILIKE '%enterococcus%'
        OR downcase_name ILIKE '%difficile%'
        OR downcase_name ILIKE '%escherichia%'
        OR downcase_name ILIKE '%campylobacter%'
        OR downcase_name ILIKE '%candida%'
        OR downcase_name ILIKE '%mrsa%'
        OR downcase_name ILIKE '%malaria%'
        OR downcase_name ILIKE '%infection%'
        OR downcase_name ILIKE '%bacterial%'
        OR downcase_name ILIKE '%fungal%'
        OR downcase_name ILIKE '%antimicrobial%'
        OR downcase_name ILIKE '%antifungal%'
        OR downcase_name ILIKE '%antibiotic%'
    );""")
    
    count = pd.read_sql(query, conn)
print(count)

                                        downcase_name  trial_count
0         methicillin-resistant staphylococcus aureus           88
1                   tuberculosis, multidrug-resistant           72
2                            antimicrobial resistance           46
3   carbapenem-resistant enterobacteriaceae infection           44
4                         multi-antibiotic resistance           36
5                         drug-resistant tuberculosis           30
6                   multi-drug resistant tuberculosis           30
7                    multidrug resistant tuberculosis           30
8                               antibiotic resistance           28
9             extensively drug-resistant tuberculosis           16
10                     antimicrobial resistance (amr)           12
11                  multi drug resistant tuberculosis           12
12            multidrug resistant bacterial infection           12
13                   multidrug-resistant tuberculosis         

This ended up looking much closer to the cohort we are seeking. The problem is sometimes certain terms do not show up on the same trial line so we should split the logic so that if it has "resistance" type terms of "antibiotic" on another line of the trial this is still counted. I will review if this seems to give us bad results but I think this is our best bet.

In [28]:
with engine.connect() as conn:
    query = text("""SELECT COUNT(DISTINCT s.nct_id)
FROM ctgov.studies s
WHERE 
    EXISTS (
        SELECT 1 FROM ctgov.conditions c 
        WHERE c.nct_id = s.nct_id
        AND (
            c.downcase_name ILIKE '%-resistant%'
            OR c.downcase_name ILIKE '%drug resistant%'
            OR c.downcase_name ILIKE '%antimicrobial resistance%'
            OR c.downcase_name ILIKE '%antibiotic resistance%'
        )
    )
    AND EXISTS (
        SELECT 1 FROM ctgov.conditions c 
        WHERE c.nct_id = s.nct_id
        AND (
            c.downcase_name ILIKE '%staphylococcus%'
            OR c.downcase_name ILIKE '%tuberculosis%'
            OR c.downcase_name ILIKE '%enterobacteriaceae%'
            OR c.downcase_name ILIKE '%pneumoniae%'
            OR c.downcase_name ILIKE '%pseudomonas%'
            OR c.downcase_name ILIKE '%klebsiella%'
            OR c.downcase_name ILIKE '%acinetobacter%'
            OR c.downcase_name ILIKE '%enterococcus%'
            OR c.downcase_name ILIKE '%difficile%'
            OR c.downcase_name ILIKE '%escherichia%'
            OR c.downcase_name ILIKE '%campylobacter%'
            OR c.downcase_name ILIKE '%candida%'
            OR c.downcase_name ILIKE '%mrsa%'
            OR c.downcase_name ILIKE '%malaria%'
            OR c.downcase_name ILIKE '%infection%'
            OR c.downcase_name ILIKE '%bacterial%'
            OR c.downcase_name ILIKE '%fungal%'
            OR c.downcase_name ILIKE '%antimicrobial%'
            OR c.downcase_name ILIKE '%antifungal%'
            OR c.downcase_name ILIKE '%antibiotic%'
            OR c.downcase_name ILIKE '%gonorrhoeae%'
            OR c.downcase_name ILIKE '%salmonella%'
            OR c.downcase_name ILIKE '%streptococcus%'
            OR c.downcase_name ILIKE '%helicobacter%'
        )
    );""")

    test = pd.read_sql(query, conn)
print(test)

   count
0    313


313 sounds like a much more reasonable number. We should check some of the conditions to make sure they make sense. 

In [26]:
with engine.connect() as conn:
    query = text("""    
    SELECT DISTINCT downcase_name, COUNT(*) as trial_count
    FROM ctgov.conditions
    WHERE nct_id IN (
    SELECT DISTINCT s.nct_id
    FROM ctgov.studies s
    WHERE 
        EXISTS (
            SELECT 1 FROM ctgov.conditions c2
            WHERE c2.nct_id = s.nct_id
            AND (
                c2.downcase_name ILIKE '%-resistant%'
                OR c2.downcase_name ILIKE '%drug resistant%'
                OR c2.downcase_name ILIKE '%antimicrobial resistance%'
                OR c2.downcase_name ILIKE '%antibiotic resistance%'
            )
        )
        AND EXISTS (
            SELECT 1 FROM ctgov.conditions c3
            WHERE c3.nct_id = s.nct_id
            AND (
                c3.downcase_name ILIKE '%staphylococcus%'
                OR c3.downcase_name ILIKE '%tuberculosis%'
                OR c3.downcase_name ILIKE '%mrsa%'
                OR c3.downcase_name ILIKE '%bacterial%'
                OR c3.downcase_name ILIKE '%antimicrobial%'
                OR c3.downcase_name ILIKE '%antibiotic%'
            )
        )
)
GROUP BY downcase_name
ORDER BY trial_count DESC
LIMIT 50;""")

    test = pd.read_sql(query, conn)
print(test)    

                                        downcase_name  trial_count
0         methicillin-resistant staphylococcus aureus           88
1                   tuberculosis, multidrug-resistant           72
2                                        tuberculosis           52
3                            antimicrobial resistance           46
4                         multi-antibiotic resistance           36
5                         drug-resistant tuberculosis           30
6                   multi-drug resistant tuberculosis           30
7                    multidrug resistant tuberculosis           30
8                               antibiotic resistance           28
9                             tuberculosis, pulmonary           28
10            extensively drug-resistant tuberculosis           16
11                     antimicrobial resistance (amr)           12
12  carbapenem-resistant enterobacteriaceae infection           12
13                  multi drug resistant tuberculosis         

For the most part these make sense as being either directly related, or within a trial adjacent to an AMR study.

In [ ]:
with engine.connect() as conn:
    query = text("""
    SELECT 
    s.nct_id,
    s.brief_title,
    s.overall_status,
    s.phase,
    s.start_date,
    s.completion_date,
    s.enrollment,
    sp.name AS sponsor_name,
    sp.lead_or_collaborator,
    EXTRACT(YEAR FROM s.start_date) AS start_year,
    CASE 
        WHEN s.start_date < '2015-01-01' THEN 'Pre-WHO GAP'
        WHEN s.start_date >= '2015-01-01' THEN 'Post-WHO GAP'
        ELSE 'Unknown'
    END AS who_gap_period,
    c.downcase_name AS condition
FROM ctgov.studies s
JOIN ctgov.sponsors sp ON s.nct_id = sp.nct_id
JOIN ctgov.conditions c ON s.nct_id = c.nct_id
WHERE 
    EXISTS (
        SELECT 1 FROM ctgov.conditions c2
        WHERE c2.nct_id = s.nct_id
        AND (
            c2.downcase_name ILIKE '%-resistant%'
            OR c2.downcase_name ILIKE '%drug resistant%'
            OR c2.downcase_name ILIKE '%antimicrobial resistance%'
            OR c2.downcase_name ILIKE '%antibiotic resistance%'
        )
    )
    AND EXISTS (
        SELECT 1 FROM ctgov.conditions c3
        WHERE c3.nct_id = s.nct_id
        AND (
            c3.downcase_name ILIKE '%staphylococcus%'
            OR c3.downcase_name ILIKE '%tuberculosis%'
            OR c3.downcase_name ILIKE '%enterobacteriaceae%'
            OR c3.downcase_name ILIKE '%pseudomonas%'
            OR c3.downcase_name ILIKE '%pneumoniae%'
            OR c3.downcase_name ILIKE '%klebsiella%'
            OR c3.downcase_name ILIKE '%acinetobacter%'
            OR c3.downcase_name ILIKE '%enterococcus%'
            OR c3.downcase_name ILIKE '%difficile%'
            OR c3.downcase_name ILIKE '%escherichia%'
            OR c3.downcase_name ILIKE '%campylobacter%'
            OR c3.downcase_name ILIKE '%candida%'
            OR c3.downcase_name ILIKE '%mrsa%'
            OR c3.downcase_name ILIKE '%malaria%'
            OR c3.downcase_name ILIKE '%infection%'
            OR c3.downcase_name ILIKE '%bacterial%'
            OR c3.downcase_name ILIKE '%fungal%'
            OR c3.downcase_name ILIKE '%antimicrobial%'
            OR c3.downcase_name ILIKE '%antifungal%'
            OR c3.downcase_name ILIKE '%antibiotic%'
            OR c3.downcase_name ILIKE '%gonorrhoeae%'
            OR c3.downcase_name ILIKE '%salmonella%'
            OR c3.downcase_name ILIKE '%streptococcus%'
            OR c3.downcase_name ILIKE '%helicobacter%'
        )
    )
ORDER BY s.start_date;
    """)
    
    df = pd.read_sql(query, conn)
#Final Cohort Dataframe Save
os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/amr_cohort.csv', index=False)
print(f"Saved {len(df)} rows to data/processed/amr_cohort.csv")
print(f"Unique trials: {df['nct_id'].nunique()}")


Saved 8476 rows to data/processed/amr_cohort.csv
Unique trials: 313


Now that we are confident with the cohort, I am going to run through some more data checks to look at the different groups and where data may need to be cleaned up. 

In [31]:
#Number before
print(f"Number of pre 2015 studies: {len(df[df['who_gap_period'] == 'Pre-WHO GAP'])}")
print(f"Number of post 2015 studies: {len(df[df['who_gap_period'] == 'Post-WHO GAP'])}")



Number of pre 2015 studies: 1836
Number of post 2015 studies: 6620


In [32]:
# Find the oldest and newest trial start years
oldest_year = df['start_year'].min()
newest_year = df['start_year'].max()
split_year = 2015

dist_oldest_to_split = split_year - oldest_year
dist_newest_to_split = newest_year - split_year

print(f"Oldest trial year: {oldest_year}")
print(f"Newest trial year: {newest_year}")
print(f"Distance from oldest to split (2015): {dist_oldest_to_split} years")
print(f"Distance from split (2015) to newest: {dist_newest_to_split} years")

if dist_oldest_to_split == dist_newest_to_split:
    print("The distances are equal.")
else:
    print("The distances are NOT equal.")


#Finding the number of trials in each period
print(f"\nTotal trials: {len(df)}")
print(f"Date range: {oldest_year} to {newest_year}")
print(f"\nPre/Post WHO GAP split:")
print(df['who_gap_period'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum())




Oldest trial year: 2004.0
Newest trial year: 2026.0
Distance from oldest to split (2015): 11.0 years
Distance from split (2015) to newest: 11.0 years
The distances are equal.

Total trials: 8476
Date range: 2004.0 to 2026.0

Pre/Post WHO GAP split:
who_gap_period
Post-WHO GAP    6620
Pre-WHO GAP     1836
Unknown           20
Name: count, dtype: int64

Missing values:
nct_id                     0
brief_title                0
overall_status             0
phase                   3480
start_date                20
completion_date           60
enrollment                16
sponsor_name               0
lead_or_collaborator       0
start_year                20
who_gap_period             0
condition                  0
dtype: int64


In [33]:
# Phase distribution
print("Phase distribution:")
print(df['phase'].value_counts())

print("\nTop 10 conditions:")
print(df['condition'].value_counts().head(10))

print("\nSponsor type breakdown:")
print(df['lead_or_collaborator'].value_counts())

Phase distribution:
phase
NA               1780
PHASE3            796
PHASE2/PHASE3     620
PHASE2            560
PHASE4            548
PHASE1            492
PHASE1/PHASE2     108
EARLY_PHASE1       92
Name: count, dtype: int64

Top 10 conditions:
condition
tuberculosis, multidrug-resistant                    584
methicillin-resistant staphylococcus aureus          348
tuberculosis                                         284
antimicrobial resistance                             216
multi-drug resistant tuberculosis                    200
drug-resistant tuberculosis                          192
extensively drug-resistant tuberculosis              184
carbapenem-resistant enterobacteriaceae infection    164
tuberculosis, pulmonary                              156
antibiotic resistance                                132
Name: count, dtype: int64

Sponsor type breakdown:
lead_or_collaborator
collaborator    4884
lead            3592
Name: count, dtype: int64
